# Exploratory Data Analysis — MovieLens 1M

This notebook explores the preprocessed MovieLens 1M dataset, visualizing rating distributions and user activity. It also provides a summary of model metrics and displays an example of the SHAP-based explanation generated by the recommendation system.

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
try:
    train_df = pd.read_csv('../data/processed/train.csv')
    test_df = pd.read_csv('../data/processed/test.csv')
    with open('../data/processed/item_names.json', 'r') as f:
        item_names = json.load(f)
        
    unique_users = train_df['user_idx'].nunique()
    unique_items = train_df['item_idx'].nunique()
    total_train = len(train_df)
    total_test = len(test_df)
    sparsity = 1.0 - ((total_train + total_test) / (unique_users * unique_items))
    
    print(f"Total Users: {unique_users}")
    print(f"Total Items: {unique_items}")
    print(f"Total Train Ratings: {total_train}")
    print(f"Total Test Ratings: {total_test}")
    print(f"Matrix Sparsity: {sparsity:.4%}")
except FileNotFoundError:
    print("Data files not found. Please run src/prepare_data.py first.")

In [ ]:
if 'train_df' in locals():
    plt.figure(figsize=(8, 5))
    plt.hist(train_df['rating'], bins=5, edgecolor='black', alpha=0.7)
    plt.title("Rating Distribution — Training Set")
    plt.xlabel("Rating")
    plt.ylabel("Count")
    plt.show()

In [ ]:
if 'train_df' in locals():
    user_counts = train_df['user_idx'].value_counts()
    plt.figure(figsize=(8, 5))
    plt.hist(user_counts, bins=50, edgecolor='black', alpha=0.7)
    plt.title("Ratings per User — Training Set")
    plt.xlabel("Number of Ratings")
    plt.ylabel("User Count")
    plt.show()

In [ ]:
if 'train_df' in locals() and 'item_names' in locals():
    item_counts = train_df['item_idx'].value_counts().reset_index()
    item_counts.columns = ['item_idx', 'rating_count']
    item_counts['title'] = item_counts['item_idx'].apply(lambda x: item_names.get(str(x), 'Unknown'))
    display(item_counts.head(20))

## Model Metrics

In [ ]:
model_metrics_path = '../results/model_metrics.json'
baseline_metrics_path = '../results/baseline_metrics.json'

if os.path.exists(model_metrics_path) and os.path.exists(baseline_metrics_path):
    with open(model_metrics_path, 'r') as f:
        model_metrics = json.load(f)
    with open(baseline_metrics_path, 'r') as f:
        baseline_metrics = json.load(f)
        
    df = pd.DataFrame([baseline_metrics, model_metrics], index=['Popularity Baseline', 'NeuMF'])
    display(df)
else:
    print("Run src/train_model.py first.")

## Example Explanation

In [ ]:
explanation_path = '../results/explanation.json'
if os.path.exists(explanation_path):
    with open(explanation_path, 'r') as f:
        explanation = json.load(f)
    print("Human Readable Explanation:")
    print(explanation.get('human_readable_explanation', 'N/A'))
    print("\nTop Contributors:")
    df_exp = pd.DataFrame(explanation.get('top_contributors', []))
    display(df_exp)
else:
    print("Run src/generate_explanation.py first.")